## Import And Clean Data

In [1]:
import pandas as pd 
import numpy as np 
dataset = pd.read_csv('Data.csv')
#***************************************************
dataset = dataset [dataset['price']!=0]
dataset = dataset[(dataset['bedrooms'] != 0) & (dataset['bathrooms'] != 0)]
dataset = dataset.drop(dataset.nlargest(2, "price").index)
dataset = dataset.drop(index=4351)
#print(dataset['price'].max())
#print(dataset['price'].min())
#print(dataset['price'].mean())
#print(dataset['price'].median())
#***************************************************
X = dataset.drop(columns=["price", "date", "street", "country"])
y = dataset["price"]

## Encoding Categorical Features

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(), ['city', 'statezip'])],
    remainder='passthrough'
)
X_encoded = ct.fit_transform(X)

## Train-Test Split And Feature Scaling

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( X_encoded, y, test_size=0.25, random_state=0)

from sklearn.preprocessing import StandardScaler
X_train = X_train.toarray()
X_test = X_test.toarray()

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Model Comparison (Cross-validation)

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error

y_train_log = np.log1p(y_train)

from sklearn.model_selection import cross_val_score

models = {
    "Linear": LinearRegression(),
    "SVR": SVR(),
    "RandomForest": RandomForestRegressor(random_state=0),
    "XGBoost": XGBRegressor(random_state=0),    
}

for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train_log, cv=5, scoring="r2")
    print(name, "R2 (log):", round(scores.mean(), 4), "+/-", round(scores.std(), 4))

Linear R2 (log): 0.7985 +/- 0.023
SVR R2 (log): 0.7944 +/- 0.0219
RandomForest R2 (log): 0.7466 +/- 0.0199
XGBoost R2 (log): 0.7721 +/- 0.034


## SVR Tuning & Final Evaluation

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {"C": [1, 10, 100], "epsilon": [0.01, 0.1]}

grid = GridSearchCV(SVR(), param_grid, cv=5, scoring="r2")
grid.fit(X_train, y_train_log)
print(grid.best_params_, grid.best_score_)

best_svr = grid.best_estimator_
y_pred_best = np.expm1(best_svr.predict(X_test))
print("R2:", r2_score(y_test, y_pred_best))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best)))

{'C': 1, 'epsilon': 0.1} 0.7944484355236685
R2: 0.7690161571166412
RMSE: 176930.3501653435


## Save the Model

In [ ]:
import joblib

joblib.dump(ct, "ct.joblib")
joblib.dump(sc, "scaler.joblib")
joblib.dump(best_svr, "svr_model.joblib")